# NB54: Spark + MinIO

Writing streaming data to a MinIO Data Lake.

## 1. Environment Setup

This cell installs **Java 8**, **Spark 3.5.0**, **Kafka 3.6.1**, and necessary Python libraries (`pyspark`, `kafka-python`, `redis`, `pymongo`, `elasticsearch`, `cassandra-driver`, `minio`). It also sets environment variables for Java and Spark.

In [ ]:
# Install Dependencies
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/spark/spark-3.5.0/spark-3.5.0-bin-hadoop3.tgz
!tar xf spark-3.5.0-bin-hadoop3.tgz
!wget -q https://archive.apache.org/dist/kafka/3.6.1/kafka_2.13-3.6.1.tgz
!tar xf kafka_2.13-3.6.1.tgz
!pip uninstall -y numpy
!pip install -q "numpy<2.0.0"
!pip install -q findspark pyspark kafka-python redis pymongo elasticsearch==7.10.1 cassandra-driver minio

# Environment Variables
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.0-bin-hadoop3"
import findspark
findspark.init()

## 2. Start Services

This cell starts the required distributed services in the background:
- **Kafka & Zookeeper**: Event streaming platform.
- **MinIO**: S3-compatible object storage.

In [ ]:
# Start Kafka
!./kafka_2.13-3.6.1/bin/zookeeper-server-start.sh -daemon ./kafka_2.13-3.6.1/config/zookeeper.properties
!./kafka_2.13-3.6.1/bin/kafka-server-start.sh -daemon ./kafka_2.13-3.6.1/config/server.properties
# Start MinIO
!wget -q https://dl.min.io/server/minio/release/linux-amd64/minio
!chmod +x minio
!mkdir -p /content/minio_data
!MINIO_ROOT_USER=minioadmin MINIO_ROOT_PASSWORD=minioadmin ./minio server /content/minio_data --console-address ":9001" &> minio.log &

import time, socket, os
def wait_for_port(port, host='localhost', timeout=120):
    start_time = time.time()
    while True:
        try:
            with socket.create_connection((host, port), timeout=1):
                print(f"Service at {host}:{port} is ready!")
                return True
        except (OSError, ConnectionRefusedError):
            if time.time() - start_time > timeout:
                print(f"Timeout waiting for {host}:{port} to start.")
                # Dump logs for debugging
                if os.path.exists('minio.log'):
                    print('--- MINIO LOG ---')
                    print(open('minio.log').read())
                if os.path.exists('es.log'):
                    print('--- ES LOG ---')
                    print(open('es.log').read())
                if os.path.exists('cassandra.log'):
                    print('--- CASSANDRA LOG ---')
                    print(open('cassandra.log').read())
                raise Exception(f"Service at {host}:{port} failed to start.")
            time.sleep(2)

# Wait for services
wait_for_port(9092) # Kafka
wait_for_port(9000) # MinIO
time.sleep(5) # Extra buffer for MinIO


## 3. Create Kafka Topic

Creates a topic named `input-topic` with 1 partition and replication factor 1.

In [ ]:
# Create Topic
!./kafka_2.13-3.6.1/bin/kafka-topics.sh --create --topic input-topic --bootstrap-server localhost:9092 --replication-factor 1 --partitions 1

## 4. Producer

Sends file data chunks to Kafka.

In [ ]:
from kafka import KafkaProducer
import time
print("Starting Producer...")
producer = KafkaProducer(bootstrap_servers='localhost:9092')
print("Sending 100 data chunks...")
for i in range(100): producer.send('input-topic', f'data_{i}'.encode('utf-8'))
producer.flush()
print("Producer finished.")

## 5. Spark -> MinIO (Data Lake)

Aggregates messages in a batch and uploads them as a file to MinIO bucket `spark-bucket`.

In [ ]:
%%writefile kafka_consumer.py
from pyspark.sql import SparkSession
from minio import Minio
import io

m_client = Minio("127.0.0.1:9000", access_key="minioadmin", secret_key="minioadmin", secure=False)
if not m_client.bucket_exists("spark-bucket"): m_client.make_bucket("spark-bucket")

spark = SparkSession.builder.appName("MinIO").getOrCreate()

def process_batch(df, epoch_id):
    val = "\n".join([r.value.decode('utf-8') for r in df.collect()])
    if val:
        m_client.put_object("spark-bucket", f"batch_{epoch_id}.txt", io.BytesIO(val.encode('utf-8')), len(val))
    print(f"Batch {epoch_id} uploaded to MinIO.")

print("Starting Spark Streaming Job...")
df = spark.readStream.format("kafka").option("kafka.bootstrap.servers", "localhost:9092").option("subscribe", "input-topic").option("startingOffsets", "earliest").load()
query = df.selectExpr("CAST(value AS STRING)").writeStream.foreachBatch(process_batch).start()
query.awaitTermination(30)
print("Spark Job Finished.")

In [ ]:
!spark-submit --packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0 kafka_consumer.py

## 6. Verification

List objects in MinIO bucket.

In [ ]:
from minio import Minio
m_client = Minio("127.0.0.1:9000", access_key="minioadmin", secret_key="minioadmin", secure=False)
print("Listing objects in MinIO bucket...")
objects = m_client.list_objects("spark-bucket")
print("--- Files in MinIO ---")
for obj in objects:
    print(obj.object_name)